# Tropical Forest Carbon Stability Analysis

**Question:** Do functional traits mediate the effect of compound water stress (VPD + soil moisture deficit) on temporal stability of forest aboveground carbon?

**Scope:** ~1000 stratified points in *Tropical & Subtropical Moist Broadleaf Forests*

**Data:**
- CTrees AGB 100 m (2000–2025) → stability = mean(AGB) / SD(AGB)
- TerraClimate → VPD, soil moisture deficit, water deficit, PDSI
- Lusk et al. 2026 trait maps (sPlot-based) → SSD, SLA, rooting depth, etc.

**Run in Google Colab** (recommended for Earth Engine authentication).

## 1. Install dependencies

In [ ]:
!pip install -q earthengine-api geemap pandas numpy matplotlib statsmodels scikit-learn

## 2. Authenticate Earth Engine

Run once per Colab session. Follow the popup link and paste the verification code.

In [ ]:
import ee
import geemap

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

print('Earth Engine initialized')

## 3. Extract ~5000 candidate points from GEE

This samples at **1 km** over tropical moist forest, extracts AGB stability, climate compounds, and traits.

> **Note:** If `getInfo()` times out, use `export_to_drive()` in `gee_extract.py` or the `gee_extract.js` script in the Earth Engine Code Editor, then upload the CSV to Colab.

In [ ]:
import sys
sys.path.insert(0, 'tropical_forest_stability')

from pathlib import Path
import pandas as pd

from gee_extract import (
    initialize_gee,
    get_tropical_moist_forest_geometry,
    build_analysis_image,
    sample_candidate_points,
    features_to_records,
    export_to_drive,
)
from config import OUTPUT_DIR, POINTS_RAW_CSV, N_CANDIDATE_POINTS

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

initialize_gee()
region = get_tropical_moist_forest_geometry()
image = build_analysis_image()
fc = sample_candidate_points(image, region, n_points=N_CANDIDATE_POINTS)

# Option A: download in memory (works for ~5000 points)
records = features_to_records(fc)
df_raw = pd.DataFrame(records)
df_raw.to_csv(POINTS_RAW_CSV, index=False)
print(f'Saved {len(df_raw)} raw points to {POINTS_RAW_CSV}')
display(df_raw.head())

# Option B (uncomment if Option A fails):
# task = export_to_drive(fc, description='tropical_moist_forest_points_raw')
# print('Export started. Download CSV from Google Drive, then set POINTS_RAW_CSV path.')

## 4. Quick map check (optional)

In [ ]:
import ee
from gee_extract import build_agb_stability_image, build_agb_collection, get_tropical_moist_forest_geometry

stability = build_agb_stability_image(build_agb_collection()).select('stability_mu_sigma')
region = get_tropical_moist_forest_geometry()

m = geemap.Map(center=[0, -60], zoom=4)
m.add_layer(stability.clip(region), {'min': 0, 'max': 15, 'palette': ['red', 'yellow', 'green']}, 'AGB stability')
m

## 5. Stratified sample to 1000 points + mediation analysis

In [ ]:
from analysis import run_full_analysis, load_and_clean, stratified_sample
from config import POINTS_RAW_CSV

sampled, results = run_full_analysis(raw_csv=POINTS_RAW_CSV)
display(sampled.describe())
display(results)

## 6. Individual mediation: VPD and soil moisture separately

In [ ]:
from analysis import run_mediation, zscore
import pandas as pd

df = load_and_clean(POINTS_RAW_CSV)
df = stratified_sample(df, n=1000)

mediators = ['ssd_g_cm3', 'rooting_depth_m', 'sla_m2_kg']
rows = []

for exposure in ['vpd_p95', 'soil_deficit']:
    for med in mediators:
        if med in df.columns:
            r = run_mediation(df, exposure=exposure, mediator=med, covariates=['pr_cv'])
            rows.append(r.__dict__)

pd.DataFrame(rows)

## 7. View figures

In [ ]:
from IPython.display import Image, display
from config import FIGURES_DIR

display(Image(filename=f'{FIGURES_DIR}/diagnostic_scatter.png'))
display(Image(filename=f'{FIGURES_DIR}/mediation_indirect_effects.png'))

## 8. If you already have a CSV from GEE export

Skip sections 2–3 and run only:

```python
from analysis import run_full_analysis
run_full_analysis(raw_csv='path/to/your_export.csv')
```